# Assignment 2 — 序列语言模型（GPU 版，Cosmopedia 直连）
**Author:** _Zhang Jingxuan_    **Date:** 2025-09-10

- 强制使用 **HuggingFace Cosmopedia**（或其 100k 子集）作为原始语料，不再使用任何示例文本；
- 自动优先选择 **GPU (cuda)**，检测不到时回落到 **cpu**；
- 内置 DataLoader 与 CUDA 优化（pin_memory / num_workers / TF32）。


## 1. 环境与设备（优先 GPU）

In [1]:
import sys, os, math, time, random, re
from dataclasses import dataclass
from typing import List, Tuple, Dict, Any
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
if device == "cuda":
    try:
        print("GPU:", torch.cuda.get_device_name(0))
        # CUDA 性能小优化
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.backends.cudnn.benchmark = True
    except Exception as e:
        print("GPU info error:", e)

Python: 3.9.19 (main, May  6 2024, 20:12:36) [MSC v.1916 64 bit (AMD64)]
PyTorch: 2.5.1+cu121
CUDA available: True
Using device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


## 2. 从 HuggingFace Cosmopedia 加载语料（强制使用）
> 默认启用 **streaming**，并通过条数/字符上限控制规模。可先用 `cosmopedia-100k` 做流程验证。


In [2]:
# 如需安装：取消下一行注释
# %pip -q install datasets huggingface_hub

from datasets import load_dataset
from pathlib import Path

# === 你可以在此处按需修改 ===
HF_DATASET_NAME = "HuggingFaceTB/cosmopedia"      # 也可改为 "HuggingFaceTB/cosmopedia-100k"
HF_CONFIG = "stories"                              # auto_math_text / khanacademy / openstax / stanford / stories / web_samples_v1 / web_samples_v2 / wikihow
HF_SPLIT = "train"
HF_STREAMING = True                                # 大数据建议 True
HF_SAMPLE_LIMIT = 5000                             # 流式读取最大样本条数（先小跑；跑大点可酌情增大）
HF_MAX_CHARS = 2_000_000                           # 拼接为纯文本的最大字符数（控制内存）
HF_TEXT_FIELD = "text"
HF_FILTER_FORMAT = None                            # 例: 只保留教材风格 -> "textbook"

def load_corpus_from_hf(
    dataset_name=HF_DATASET_NAME, config=HF_CONFIG, split=HF_SPLIT,
    streaming=HF_STREAMING, sample_limit=HF_SAMPLE_LIMIT, max_chars=HF_MAX_CHARS,
    text_field=HF_TEXT_FIELD, filter_format=HF_FILTER_FORMAT
):
    if streaming:
        ds = load_dataset(dataset_name, config, split=split, streaming=True)
        total_chars, total_rows = 0, 0
        texts = []
        for ex in ds:
            if filter_format is not None and ex.get("format", None) != filter_format:
                continue
            t = ex.get(text_field, None)
            if not t:
                continue
            texts.append(t.strip())
            total_rows += 1
            total_chars += len(t)
            if sample_limit is not None and total_rows >= sample_limit:
                break
            if max_chars is not None and total_chars >= max_chars:
                break
        raw_text = "\n\n".join(texts)
        return raw_text, dict(rows=total_rows, chars=len(raw_text), streaming=True)
    else:
        ds = load_dataset(dataset_name, config, split=split)
        if filter_format is not None and "format" in ds.column_names:
            ds = ds.filter(lambda ex: ex.get("format", None) == filter_format)
        if sample_limit is not None:
            ds = ds.select(range(min(sample_limit, len(ds))))
        texts = [t.strip() for t in ds[text_field] if isinstance(t, str)]
        raw_text = "\n\n".join(texts)
        if max_chars is not None and len(raw_text) > max_chars:
            raw_text = raw_text[:max_chars]
        return raw_text, dict(rows=len(texts), chars=len(raw_text), streaming=False)

print("Loading Cosmopedia:", HF_DATASET_NAME, "| config:", HF_CONFIG, "| split:", HF_SPLIT)
raw_text, info = load_corpus_from_hf()
print(f"Fetched rows={info['rows']}  chars={info['chars']}  streaming={info['streaming']}")

# 持久化（便于复现实验） —— 保存到当前目录下的 data/ 文件夹
save_dir = Path.cwd() / "data"
save_dir.mkdir(parents=True, exist_ok=True)  # 自动创建 data 目录
out_txt = save_dir / "cosmopedia_corpus.txt"
out_txt.write_text(raw_text, encoding="utf-8")
print("Saved concatenated corpus to:", out_txt.resolve())

Loading Cosmopedia: HuggingFaceTB/cosmopedia | config: stories | split: train


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/43 [00:00<?, ?it/s]

Fetched rows=741  chars=2000903  streaming=True
Saved concatenated corpus to: D:\Coding_Nus\DSA4213\DSA4213_Natural-Language-Processing-for-Data-Science\data\cosmopedia_corpus.txt


## 3. 训练配置与消融表（GPU 友好默认）

In [3]:
import pandas as pd

DEFAULTS = dict(
    tokenizer="char",   # "char" | "word"
    seq_len=256,
    emb_dim=256,
    hidden=256,
    num_layers=1,
    nhead=4,
    tfm_layers=2,
    dropout=0.1,
    batch_size=32,
    epochs=3,
    lr=3e-4,
    clip=1.0,
    seed=42,
    device=("cuda" if torch.cuda.is_available() else "cpu"),
)

# 你可以按需调整（建议先小跑验证流程）
ABLATIONS = [
    dict(model="lstm",        tokenizer="char", seq_len=256, dropout=0.2, epochs=3, batch_size=32),
    dict(model="transformer", tokenizer="char", seq_len=256, dropout=0.1, epochs=3, batch_size=16),
    dict(model="transformer", tokenizer="char", seq_len=128, dropout=0.1, epochs=3, batch_size=32),
    dict(model="lstm",        tokenizer="word", seq_len=128, dropout=0.2, epochs=3, batch_size=32),
]
OUT_DIR = Path("runs_nb_gpu"); OUT_DIR.mkdir(exist_ok=True, parents=True)

# DataLoader 优化参数（Windows 下多进程需注意；若遇到问题把 num_workers=0）
NUM_WORKERS = 2 if torch.cuda.is_available() else 0
PIN_MEMORY  = True if torch.cuda.is_available() else False

## 4. 工具函数

In [4]:
def set_seed(seed: int = 42):
    import random, numpy as np, torch
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def save_plot(train_losses, val_losses, out_png):
    plt.figure()
    plt.plot(train_losses, label="train")
    plt.plot(val_losses, label="val")
    plt.xlabel("Epoch"); plt.ylabel("Loss")
    plt.legend(); plt.title("Training / Validation Loss")
    plt.tight_layout(); plt.savefig(out_png); plt.show()

## 5. 分词器

In [5]:
class CharTokenizer:
    def __init__(self, text: str):
        chars = sorted(list(set(text)))
        self.stoi = {c:i for i,c in enumerate(chars)}
        self.itos = {i:c for c,i in self.stoi.items()}
        self.vocab_size = len(self.stoi)
    def encode(self, text: str): return [self.stoi[c] for c in text if c in self.stoi]
    def decode(self, ids): return "".join(self.itos[i] for i in ids)

import re as _re
class WordTokenizer:
    WORD_RE = _re.compile(r"\w+|[^\w\s]", _re.UNICODE)
    def __init__(self, text: str, min_freq: int = 1, max_vocab: int = 50000):
        toks = self.WORD_RE.findall(text); freq = {}
        for t in toks: freq[t] = freq.get(t, 0) + 1
        items = sorted(freq.items(), key=lambda x:(-x[1], x[0]))[:max_vocab]
        self.stoi = {tk:i for i,(tk,cnt) in enumerate(items) if cnt >= min_freq}
        self.itos = {i:tk for tk,i in self.stoi.items()}
        self.unk = len(self.stoi); self.stoi["<UNK>"]=self.unk; self.itos[self.unk]="<UNK>"
        self.vocab_size = len(self.stoi)
    def encode(self, text: str):
        return [self.stoi.get(t, self.unk) for t in self.WORD_RE.findall(text)]
    def decode(self, ids): return " ".join(self.itos[i] for i in ids)

## 6. 数据集与切块（80/10/10）

In [6]:
class LMSequenceDataset(Dataset):
    def __init__(self, ids: List[int], seq_len: int):
        self.ids = ids; self.seq_len = seq_len
        self.num_seq = (len(ids) - 1) // seq_len
    def __len__(self): return self.num_seq
    def __getitem__(self, idx):
        s = idx * self.seq_len
        x = torch.tensor(self.ids[s:s+self.seq_len], dtype=torch.long)
        y = torch.tensor(self.ids[s+1:s+self.seq_len+1], dtype=torch.long)
        return x, y

def split_data(tokens: List[int], ratios=(0.8,0.1,0.1)):
    n=len(tokens); i=int(n*ratios[0]); j=int(n*(ratios[0]+ratios[1])); return tokens[:i], tokens[i:j], tokens[j:]

## 7. 模型：LSTM LM 与 Causal Transformer LM

In [7]:
class LSTMLM(nn.Module):
    def __init__(self, vocab_size, emb_dim=256, hidden=256, num_layers=1, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden, num_layers=num_layers, batch_first=True,
                            dropout=dropout if num_layers>1 else 0.0)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, vocab_size)
    def forward(self, x, hidden=None):
        x=self.embed(x); out, hidden=self.lstm(x, hidden); out=self.drop(out); return self.fc(out), hidden

class CausalTransformer(nn.Module):
    def __init__(self, vocab_size, emb_dim=256, nhead=4, num_layers=2, dropout=0.1, max_len=1024):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, emb_dim)
        self.pos_emb = nn.Embedding(max_len, emb_dim)
        enc_layer = nn.TransformerEncoderLayer(d_model=emb_dim, nhead=nhead,
                                               dim_feedforward=4*emb_dim, dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.ln_f = nn.LayerNorm(emb_dim)
        self.fc = nn.Linear(emb_dim, vocab_size)
        self.max_len = max_len
    def forward(self, x):
        B,T=x.size(); pos=torch.arange(0,T,device=x.device).unsqueeze(0).expand(B,T)
        h=self.token_emb(x)+self.pos_emb(pos)
        mask=torch.triu(torch.ones(T,T,device=x.device)*float("-inf"), diagonal=1)
        h=self.encoder(h, mask=mask); h=self.ln_f(h); return self.fc(h)

## 8. 训练 / 评估 / 生成

In [8]:
def build_tokenizer(text: str, kind: str):
    if kind=="char": return CharTokenizer(text)
    if kind=="word": return WordTokenizer(text)
    raise ValueError("tokenizer must be 'char' or 'word'")

def eval_loader(loader, model, criterion, device):
    model.eval(); total_loss=0.0; total_tokens=0
    with torch.no_grad():
        for x,y in loader:
            x=x.to(device, non_blocking=True); y=y.to(device, non_blocking=True)
            logits = model(x)[0] if isinstance(model, LSTMLM) else model(x)
            loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
            total_loss += loss.item()*y.numel(); total_tokens += y.numel()
    avg = total_loss/total_tokens; ppl = math.exp(avg) if avg<20 else float('inf'); return avg, ppl

def sample_generate(model, tok, seq_len=256, device=device, max_new_tokens=200, temperature=1.0, start_text=""):
    model.eval()
    ids = tok.encode(start_text) if start_text else [random.randint(0, tok.vocab_size-1)]
    ids = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            x = ids[:, -seq_len:]
            logits = model(x)[0] if isinstance(model, LSTMLM) else model(x)
            logits = logits[:, -1, :] / max(1e-6, temperature)
            probs = torch.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, num_samples=1)
            ids = torch.cat([ids, nxt], dim=1)
    return tok.decode(ids[0].tolist())

def run_single_experiment(raw_text: str, cfg: Dict[str, Any], label: str):
    conf = {**DEFAULTS, **cfg}; set_seed(conf["seed"])
    tok = build_tokenizer(raw_text, conf["tokenizer"]); ids = tok.encode(raw_text)
    tr_ids, va_ids, te_ids = split_data(ids, (0.8,0.1,0.1))
    train_set=LMSequenceDataset(tr_ids, conf["seq_len"]); val_set=LMSequenceDataset(va_ids, conf["seq_len"]); test_set=LMSequenceDataset(te_ids, conf["seq_len"])
    train_loader=DataLoader(train_set, batch_size=conf["batch_size"], shuffle=True, drop_last=True,
                            num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    val_loader=DataLoader(val_set, batch_size=conf["batch_size"], shuffle=False, drop_last=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    test_loader=DataLoader(test_set, batch_size=conf["batch_size"], shuffle=False, drop_last=False,
                           num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    model = (LSTMLM(tok.vocab_size, emb_dim=conf["emb_dim"], hidden=conf["hidden"], num_layers=conf["num_layers"], dropout=conf["dropout"]).to(conf["device"])
             if conf["model"]=="lstm" else
             CausalTransformer(tok.vocab_size, emb_dim=conf["emb_dim"], nhead=conf["nhead"], num_layers=conf["tfm_layers"], dropout=conf["dropout"], max_len=max(1024, conf["seq_len"])).to(conf["device"]))
    criterion = nn.CrossEntropyLoss(); opt = torch.optim.AdamW(model.parameters(), lr=conf["lr"])
    train_losses, val_losses = [], []; t0_all=time.time()
    for ep in range(1, conf["epochs"]+1):
        model.train(); run_loss=0.0; run_tok=0; t0=time.time()
        for x,y in train_loader:
            x=x.to(conf["device"], non_blocking=True); y=y.to(conf["device"], non_blocking=True)
            opt.zero_grad(set_to_none=True)
            logits = model(x)[0] if isinstance(model, LSTMLM) else model(x)
            loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), conf["clip"]); opt.step()
            run_loss += loss.item()*y.numel(); run_tok += y.numel()
        tr_avg = run_loss/run_tok; va_avg, va_ppl = eval_loader(val_loader, model, criterion, conf["device"])
        train_losses.append(tr_avg); val_losses.append(va_avg)
        print(f"[{label}] Epoch {ep}/{conf['epochs']} train={tr_avg:.4f} val={va_avg:.4f} valPPL={va_ppl:.2f} time={time.time()-t0:.1f}s")
    total_time = time.time()-t0_all
    te_loss, te_ppl = eval_loader(test_loader, model, criterion, conf["device"])
    fig_path = OUT_DIR / f"loss_{label}.png"; save_plot(train_losses, val_losses, str(fig_path))
    gens = {T: sample_generate(model, tok, seq_len=conf["seq_len"], device=conf["device"], max_new_tokens=200, temperature=T, start_text="") for T in [0.7,1.0,1.3]}
    rec = dict(label=label, model=conf["model"], tokenizer=conf["tokenizer"], seq_len=conf["seq_len"], dropout=conf["dropout"], epochs=conf["epochs"],
               batch_size=conf["batch_size"], lr=conf["lr"], emb_dim=conf["emb_dim"], val_loss=val_losses[-1],
               val_ppl=math.exp(val_losses[-1]) if val_losses[-1] < 20 else float('inf'), test_loss=te_loss, test_ppl=te_ppl, train_time_sec=total_time, curve=str(fig_path))
    for T, txt in gens.items(): rec[f"gen_T{T}"] = txt[:500]
    return rec, train_losses, val_losses, gens

## 9. 运行多组实验并汇总

In [ ]:
summary_rows = []
all_generations = {}
for i, cfg in enumerate(ABLATIONS, start=1):
    label = f"exp{i}_{cfg.get('model')}_{cfg.get('tokenizer')}_L{cfg.get('seq_len')}_d{cfg.get('dropout')}"
    print("\n==== Running", label, "====")
    rec, tr, va, gens = run_single_experiment(raw_text, cfg, label)
    summary_rows.append(rec)
    all_generations[label] = gens

import pandas as pd
from IPython.display import display
from pathlib import Path

# 结果汇总并排序
df_results = pd.DataFrame(summary_rows).sort_values(by=["test_ppl"])

# 显示结果表（Jupyter）
print("[Table] 实验结果汇总（可下载）")
display(df_results)

# 导出 CSV（保存到 OUT_DIR 或默认 runs_nb 目录）
out_base = OUT_DIR if 'OUT_DIR' in globals() else Path("runs_nb")
out_base.mkdir(parents=True, exist_ok=True)
out_csv = out_base / "results.csv"
df_results.to_csv(out_csv, index=False)
print("Saved results to:", out_csv.resolve())

# 让 DataFrame 成为单元格最终输出
df_results


==== Running exp1_lstm_char_L256_d0.2 ====


## 10. 自动要点总结与曲线预览

In [ ]:

def auto_analyze(df: "pd.DataFrame"):
    if df.empty: return "（无结果：请先运行实验。）"
    lines=[]; best=df.iloc[0]; lines.append(f"- **最佳测试困惑度**：{best['test_ppl']:.2f}  （配置：{best['label']}）")
    for name in ["model","seq_len","dropout"]:
        g=df.groupby(name)["test_ppl"].mean().sort_values()
        lines.append(f"- **按 {name} 分组均值 Test PPL**：")
        for k,v in g.items(): lines.append(f"  - {name}={k}: {v:.2f}")
    return "\n".join(lines)

print(auto_analyze(df_results))

from PIL import Image
for p in df_results['curve']:
    try:
        display(Image.open(p))
    except Exception as e:
        print("Cannot open image:", p, e)

## 11. 文本生成（温度 0.7 / 1.0 / 1.3）

In [ ]:

def show_generations(all_generations, topk=3):
    shown=0
    for label, gens in all_generations.items():
        print("\n====", label, "====")
        for T, txt in gens.items():
            print(f"[T={T}] -> {txt[:500]}")
        shown+=1
        if shown>=topk: break

show_generations(all_generations, topk=3)